# Yoruba & Igbo Diacritizer Training Pipeline (Google Colab Version)

This notebook contains the complete pipeline for compiling the training corpora and training three different diacritizer models for **Yoruba** and **Igbo**:
1. **Yoruba Dot-Below CRF (`DiacNetYorDB`)**: A fast, character-level CRF sequence labeler targeting dot-below accents (`ọ`, `ẹ`, `ṣ`).
2. **Yoruba Full Tonal BiLSTM (`DiacNetYor`)**: A word/character BiLSTM sequence labeler targeting both dot-below and tone marks.
3. **Yoruba Transformer (`DiacNetYorX`)**: Fine-tunes `castorini/afriberta_large` as a word-level sequence labeler.
4. **Igbo Dot-Below CRF (`DiacNetIbo`)**: A fast CRF sequence labeler targeting Igbo dot-below diacritics (`ị`, `ụ`, `ọ`, `ẹ`).

---
**Tip**: Enable GPU runtime for training the BiLSTM and Transformer models. In Colab, go to **Runtime** > **Change runtime type** > Select **GPU**.

In [ ]:
# Install required libraries
!pip install -q transformers datasets sklearn-crfsuite tqdm torch unicodedata2

# Verify GPU
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
elif device == "mps":
    print("Using Apple Silicon GPU (MPS).")
else:
    print("WARNING: GPU is not active. Transformer fine-tuning will be extremely slow!")

## 1. Corpus Compilation

We stream and compile the training corpora from Hugging Face's `google/WaxalNLP`, `masakhane/masakhanews`, `menyo20k_mt` (Yoruba only), and Wikipedia.
The compiled files `yoruba_diacritizer_corpus.json` and `igbo_diacritizer_corpus.json` will be saved in the local workspace.

In [ ]:
import os
import re
import json
import unicodedata
from tqdm.notebook import tqdm
from datasets import load_dataset, Features, Value

# Yoruba: dot-below (ọ,ẹ,ṣ) and/or tone marks (acute, grave, circumflex on vowels)
YORUBA_DIACRITIC_CHARS = set("ọẹṣỌẸṢọ́ọ̀ẹ́ẹ̀")
YORUBA_DIACRITIC_PATTERN = re.compile(
    r'[ọẹṣỌẸṢ]|'
    r'[aeiouAEIOU]\u0301|'
    r'[aeiouAEIOU]\u0300|'
    r'[ọẹỌẸ]\u0301|'
    r'[ọẹỌẸ]\u0300'
)

# Igbo: dot-below only (ị, ụ, ọ, ẹ)
IGBO_DIACRITIC_PATTERN = re.compile(r'[ịụọẹỊỤỌẸ]')

def has_yoruba_diacritics(text):
    nfc = unicodedata.normalize('NFC', text)
    nfd = unicodedata.normalize('NFD', nfc)
    has_dot_below = bool(re.search(r'[ọẹṣỌẸṢịụỊỤ]', nfc))
    has_tones = '\u0301' in nfd or '\u0300' in nfd
    return has_dot_below or has_tones

def has_igbo_diacritics(text):
    nfc = unicodedata.normalize('NFC', text)
    return bool(IGBO_DIACRITIC_PATTERN.search(nfc))

def strip_all_diacritics(text):
    decomposed = unicodedata.normalize('NFD', text)
    filtered = "".join(c for c in decomposed if unicodedata.category(c) != 'Mn')
    return unicodedata.normalize('NFC', filtered)

def clean_sentence(text, min_len=10, max_len=300):
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if min_len <= len(text) <= max_len:
        return text
    return ""

def stream_waxal(config, max_sentences):
    print(f"  Streaming WaxalNLP ({config})...")
    sentences = []
    features = Features({
        'id': Value('string'), 'speaker_id': Value('string'),
        'text': Value('string'), 'locale': Value('string'),
        'gender': Value('string'),
        'audio': {'path': Value('string'), 'bytes': Value('binary')}
    })
    try:
        ds = load_dataset('google/WaxalNLP', config, streaming=True, features=features)
        for item in tqdm(ds['train'], desc=config, unit="sent"):
            s = clean_sentence(item.get('text', ''))
            if s:
                sentences.append(s)
                if len(sentences) >= max_sentences:
                    break
    except Exception as e:
        print(f"  WaxalNLP {config} error: {e}")
    return sentences

def stream_masakhanews(lang, max_sentences):
    print(f"  Streaming MasakhaNEWS ({lang})...")
    sentences = []
    try:
        ds = load_dataset('masakhane/masakhanews', lang, streaming=True)
        for split in ['train', 'validation', 'test']:
            for item in tqdm(ds[split], desc=f"news-{lang}-{split}", unit="sent"):
                text = f"{item.get('headline', '')}. {item.get('text', '')}"
                for part in re.split(r'(?<=[.!?])\s+', text):
                    s = clean_sentence(part)
                    if s:
                        sentences.append(s)
                        if len(sentences) >= max_sentences:
                            return sentences
    except Exception as e:
        print(f"  MasakhaNEWS {lang} error: {e}")
    return sentences

def stream_wikipedia(lang, max_sentences):
    print(f"  Streaming Wikipedia ({lang})...")
    sentences = []
    try:
        ds = load_dataset('wikimedia/wikipedia', f'20231101.{lang}', streaming=True)
        for item in tqdm(ds['train'], desc=f"wiki-{lang}", unit="sent"):
            for part in re.split(r'(?<=[.!?])\s+', item.get('text', '')):
                s = clean_sentence(part)
                if s:
                    sentences.append(s)
                    if len(sentences) >= max_sentences:
                        return sentences
    except Exception as e:
        print(f"  Wikipedia {lang} error: {e}")
    return sentences

def stream_menyo20k(max_sentences):
    print("  Streaming MENYO-20K (Yoruba)...")
    sentences = []
    try:
        ds = load_dataset('menyo20k_mt', streaming=True, trust_remote_code=True)
        for split in ['train', 'validation', 'test']:
            try:
                for item in tqdm(ds[split], desc=f"menyo-{split}", unit="sent"):
                    yo = item.get('translation', {}).get('yo', '')
                    s = clean_sentence(yo)
                    if s:
                        sentences.append(s)
                        if len(sentences) >= max_sentences:
                            return sentences
            except Exception:
                pass
    except Exception as e:
        print(f"  MENYO-20K error: {e}")
    return sentences

def build_diacritizer_pairs(sentences, has_diacritics_fn, min_diac_ratio=0.05):
    pairs = []
    for s in sentences:
        s = unicodedata.normalize('NFC', s)
        if not has_diacritics_fn(s):
            continue
        plain = strip_all_diacritics(s)
        if plain == s:
            continue
        diff_count = sum(1 for a, b in zip(plain, s) if a != b)
        if len(s) > 0 and diff_count / len(s) < min_diac_ratio:
            continue
        pairs.append({"plain": plain, "diacritized": s})
    return pairs

def split_corpus(pairs, train_ratio=0.8, val_ratio=0.1):
    import random
    random.seed(42)
    random.shuffle(pairs)
    n = len(pairs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return {
        "train": pairs[:train_end],
        "val": pairs[train_end:val_end],
        "test": pairs[val_end:]
    }

# Build Yoruba
print("=== Yoruba Corpus Compilation ===")
yor_raw = []
yor_raw.extend(stream_waxal('yor_tts', 10000))
if len(yor_raw) < 12000:
    yor_raw.extend(stream_menyo20k(8000))
if len(yor_raw) < 12000:
    yor_raw.extend(stream_masakhanews('yor', 5000))
if len(yor_raw) < 12000:
    yor_raw.extend(stream_wikipedia('yo', 5000))

yor_pairs = build_diacritizer_pairs(yor_raw, has_yoruba_diacritics)
seen = set()
yor_pairs = [p for p in yor_pairs if not (p['plain'] in seen or seen.add(p['plain']))]
yor_corpus = split_corpus(yor_pairs)
with open('yoruba_diacritizer_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(yor_corpus, f, indent=2, ensure_ascii=False)
print(f"Yoruba Corpus: {len(yor_corpus['train'])} train / {len(yor_corpus['val'])} val / {len(yor_corpus['test'])} test saved.\n")

# Build Igbo
print("=== Igbo Corpus Compilation ===")
ibo_raw = []
ibo_raw.extend(stream_waxal('ibo_tts', 8000))
if len(ibo_raw) < 6000:
    ibo_raw.extend(stream_masakhanews('ibo', 4000))
if len(ibo_raw) < 6000:
    ibo_raw.extend(stream_wikipedia('ig', 4000))

ibo_pairs = build_diacritizer_pairs(ibo_raw, has_igbo_diacritics)
seen = set()
ibo_pairs = [p for p in ibo_pairs if not (p['plain'] in seen or seen.add(p['plain']))]
ibo_corpus = split_corpus(ibo_pairs)
with open('igbo_diacritizer_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(ibo_corpus, f, indent=2, ensure_ascii=False)
print(f"Igbo Corpus: {len(ibo_corpus['train'])} train / {len(ibo_corpus['val'])} val / {len(ibo_corpus['test'])} test saved.")

## 2. Yoruba Dot-Below CRF Diacritizer (`DiacNetYorDB`)

This model restores *only* dot-below diacritics in Yoruba text (e.g. `ọ`, `ẹ`, `ṣ`) without predicting tonal marks. It is extremely fast and acts as a high-accuracy, low-resource baseline.

In [ ]:
import os
import re
import json
import pickle
import unicodedata
from collections import Counter
import sklearn_crfsuite

def strip_tones(text):
    decomposed = unicodedata.normalize('NFD', text)
    filtered = "".join(
        c for c in decomposed
        if unicodedata.category(c) != 'Mn' or ord(c) == 0x0323
    )
    return unicodedata.normalize('NFC', filtered)

def prepare_dot_below_pairs(pairs):
    result = []
    for pair in pairs:
        diac_full = unicodedata.normalize('NFC', pair['diacritized'])
        diac_db   = strip_tones(diac_full)
        plain     = strip_all_diacritics(diac_full)
        if plain != diac_db:
            result.append({'plain': plain, 'diacritized': diac_db})
    return result

def tokenize(text):
    return re.findall(r'\S+', text)

def word_features(words, i):
    word = words[i].lower()
    features = {
        'bias': 1.0,
        'word': word,
        'word[-2:]': word[-2:],
        'word[-3:]': word[-3:],
        'word[:2]':  word[:2],
        'word[:3]':  word[:3],
        'word.len':  str(min(len(word), 10)),
        'is_first':  i == 0,
        'is_last':   i == len(words) - 1,
        'is_upper':  words[i][0].isupper() if words[i] else False,
    }
    for j, ch in enumerate(word[:8]):
        features[f'ch[{j}]'] = ch
    for vowel in 'aeiou':
        features[f'has_{vowel}'] = vowel in word

    if i > 0:
        prev = words[i-1].lower()
        features.update({'prev_word': prev, 'prev[-2:]': prev[-2:], 'prev[:2]': prev[:2]})
    else:
        features['BOS'] = True

    if i > 1:
        features['prev2_word'] = words[i-2].lower()

    if i < len(words) - 1:
        nxt = words[i+1].lower()
        features.update({'next_word': nxt, 'next[-2:]': nxt[-2:], 'next[:2]': nxt[:2]})
    else:
        features['EOS'] = True

    if i < len(words) - 2:
        features['next2_word'] = words[i+2].lower()

    return features

def build_dataset(pairs):
    X, y = [], []
    for pair in pairs:
        plain_words = tokenize(pair['plain'])
        diac_words  = tokenize(pair['diacritized'])
        if len(plain_words) != len(diac_words):
            continue
        X.append([word_features(plain_words, i) for i in range(len(plain_words))])
        y.append(list(diac_words))
    return X, y

# Load corpus
with open("yoruba_diacritizer_corpus.json", 'r', encoding='utf-8') as f:
    corpus = json.load(f)

train_pairs = prepare_dot_below_pairs(corpus['train'])
val_pairs   = prepare_dot_below_pairs(corpus['val'])
test_pairs  = prepare_dot_below_pairs(corpus['test'])

X_train, y_train = build_dataset(train_pairs)
X_val,   y_val   = build_dataset(val_pairs)
X_test,  y_test  = build_dataset(test_pairs)

vocab = {}
for pair in train_pairs + val_pairs:
    plain_words = tokenize(pair['plain'])
    diac_words  = tokenize(pair['diacritized'])
    if len(plain_words) != len(diac_words):
        continue
    for pw, dw in zip(plain_words, diac_words):
        key = pw.lower()
        if key not in vocab:
            vocab[key] = Counter()
        vocab[key][dw.lower()] += 1
vocab_top = {k: [c for c, _ in v.most_common(3)] for k, v in vocab.items()}

print("Training Yoruba Dot-Below CRF (on 1,000-sentence subset to prevent vocab scaling slow-down)...\n")
crf_yor_db = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.05,
    c2=0.1,
    max_iterations=50,
    all_possible_transitions=False,
    verbose=True,
)
crf_yor_db.fit(X_train[:1000], y_train[:1000])

def word_accuracy(crf, X, y_true):
    y_pred = crf.predict(X)
    correct = total = 0
    for true_seq, pred_seq in zip(y_true, y_pred):
        for t, p in zip(true_seq, pred_seq):
            total += 1
            if t.lower() == p.lower():
                correct += 1
    return correct / total if total > 0 else 0

val_word = word_accuracy(crf_yor_db, X_val, y_val)
test_word = word_accuracy(crf_yor_db, X_test, y_test)
print(f"\nValidation Word Accuracy (dot-below): {val_word*100:.2f}%")
print(f"Test Word Accuracy (dot-below):       {test_word*100:.2f}%")

# Save
with open("diacnet_yor_db.pkl", 'wb') as f:
    pickle.dump({'crf': crf_yor_db, 'vocab': vocab_top}, f)
with open("diacnet_yor_db_vocab.json", 'w', encoding='utf-8') as f:
    json.dump(vocab_top, f, ensure_ascii=False, indent=2)
print("Saved model to diacnet_yor_db.pkl and vocab to diacnet_yor_db_vocab.json")

## 3. Yoruba Full Tonal BiLSTM Diacritizer (`DiacNetYor`)

This model processes characters of words using a character BiLSTM to create word embeddings, and passes those through a sentence BiLSTM, followed by a candidate classification layer.

In [ ]:
import json
import re
import math
import torch
import torch.nn as nn
import torch.optim as optim
import unicodedata
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

def tokenize(text):
    return re.findall(r'\S+', text)

# Load corpus
with open("yoruba_diacritizer_corpus.json", 'r', encoding='utf-8') as f:
    corpus = json.load(f)

# Constants
PAD_IDX    = 0
UNK_IDX    = 1
CHAR_EMB_DIM = 64
HIDDEN_DIM   = 256
BATCH_SIZE   = 64
EPOCHS       = 30
PATIENCE     = 5
LR           = 2e-3

# --- Character Decomposition & Label Encoding ---
def sentence_to_char_labels(text):
    text_nfd = unicodedata.normalize('NFD', text)
    base_chars = []
    labels = []
    for char in text_nfd:
        cat = unicodedata.category(char)
        if cat == 'Mn':
            if not base_chars:
                continue
            last_label = labels[-1]
            if char == '\u0323':
                if last_label == 0:
                    labels[-1] = 1
                elif last_label == 2:
                    labels[-1] = 4
                elif last_label == 3:
                    labels[-1] = 5
            elif char == '\u0301':
                if last_label == 0:
                    labels[-1] = 2
                elif last_label == 1:
                    labels[-1] = 4
            elif char == '\u0300':
                if last_label == 0:
                    labels[-1] = 3
                elif last_label == 1:
                    labels[-1] = 5
        else:
            base_chars.append(char)
            labels.append(0)
    return base_chars, labels

def reconstruct_text(base_chars, pred_labels):
    parts = []
    for c, l in zip(base_chars, pred_labels):
        if l == 0 or not c.isalpha():
            parts.append(c)
        elif l == 1:
            parts.append(c + '\u0323')
        elif l == 2:
            parts.append(c + '\u0301')
        elif l == 3:
            parts.append(c + '\u0300')
        elif l == 4:
            parts.append(c + '\u0323\u0301')
        elif l == 5:
            parts.append(c + '\u0323\u0300')
    return unicodedata.normalize('NFC', "".join(parts))

# --- Build Vocabulary & Candidates ---
def build_char_vocab(pairs):
    char_counter = Counter()
    for pair in pairs:
        base_chars, _ = sentence_to_char_labels(pair['diacritized'])
        for c in base_chars:
            char_counter[c] += 1
    char_vocab = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
    char_vocab.update({c: i+2 for i, c in enumerate(sorted(char_counter.keys()))})
    return char_vocab

def build_word_candidates(pairs):
    cand_counter = {}
    for pair in pairs:
        plain_words = tokenize(pair['plain'])
        diac_words  = tokenize(pair['diacritized'])
        if len(plain_words) != len(diac_words):
            continue
        for pw, dw in zip(plain_words, diac_words):
            pw_l = pw.lower()
            dw_l = dw.lower()
            if pw_l not in cand_counter:
                cand_counter[pw_l] = Counter()
            cand_counter[pw_l][dw_l] += 1
    word_candidates = {}
    for pw, counts in cand_counter.items():
        word_candidates[pw] = [c for c, _ in counts.most_common()]
    return word_candidates

char_vocab = build_char_vocab(corpus['train'] + corpus['val'])
word_candidates = build_word_candidates(corpus['train'] + corpus['val'])
print(f"Character Vocab size: {len(char_vocab)}")

# --- Dataset & DataLoader ---
class DiacNetCharDataset(Dataset):
    def __init__(self, pairs, char_vocab):
        self.samples = []
        for pair in pairs:
            base_chars, labels = sentence_to_char_labels(pair['diacritized'])
            char_ids = [char_vocab.get(c, UNK_IDX) for c in base_chars]
            self.samples.append((
                torch.tensor(char_ids, dtype=torch.long),
                torch.tensor(labels, dtype=torch.long),
                base_chars
            ))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn_char(batch):
    char_ids_list, labels_list, base_chars_list = zip(*batch)
    lengths = torch.tensor([len(ids) for ids in char_ids_list], dtype=torch.long)
    padded_chars = pad_sequence(char_ids_list, batch_first=True, padding_value=PAD_IDX)
    padded_labels = pad_sequence(labels_list, batch_first=True, padding_value=-100)
    return padded_chars, lengths, padded_labels, base_chars_list

train_ds = DiacNetCharDataset(corpus['train'], char_vocab)
val_ds   = DiacNetCharDataset(corpus['val'],   char_vocab)
test_ds  = DiacNetCharDataset(corpus['test'],  char_vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn_char)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_char)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_char)

# --- Model Architecture ---
class DiacNetCharModel(nn.Module):
    def __init__(self, char_vocab_size, emb_dim=CHAR_EMB_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(char_vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim // 2,
            bidirectional=True, batch_first=True, num_layers=2, dropout=0.3
        )
        self.classifier = nn.Linear(hidden_dim, 6)
    def forward(self, char_seqs, lengths):
        emb = self.embedding(char_seqs)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.classifier(out)

# --- Evaluation Helper with Post-Processing ---
def evaluate_char_model(model, loader, char_vocab, word_candidates):
    model.eval()
    correct_chars = total_chars = 0
    correct_words = total_words = 0
    with torch.no_grad():
        for padded_chars, lengths, labels, base_chars_batch in loader:
            padded_chars = padded_chars.to(device)
            logits = model(padded_chars, lengths)
            preds = logits.argmax(dim=-1).cpu()
            for i, slen in enumerate(lengths):
                pred_labels = preds[i, :slen].tolist()
                true_labels = labels[i, :slen].tolist()
                for p, t in zip(pred_labels, true_labels):
                    if t != -100:
                        total_chars += 1
                        if p == t:
                            correct_chars += 1
                base_chars = base_chars_batch[i]
                pred_sentence = reconstruct_text(base_chars, pred_labels)
                true_sentence = reconstruct_text(base_chars, [t for t in true_labels if t != -100])
                plain_sentence = reconstruct_text(base_chars, [0] * len(base_chars))
                
                plain_words = tokenize(plain_sentence)
                pred_words = tokenize(pred_sentence)
                true_words = tokenize(true_sentence)
                
                # Apply candidate-constrained post-processing
                corrected_words = []
                for pw, pw_pred in zip(plain_words, pred_words):
                    pw_l = pw.lower()
                    cands = word_candidates.get(pw_l, [])
                    if not cands:
                        corrected_words.append(pw_pred)
                    elif pw_pred.lower() in cands:
                        corrected_words.append(pw_pred)
                    else:
                        majority = cands[0]
                        if pw_pred and pw_pred[0].isupper():
                            majority = majority.capitalize()
                        corrected_words.append(majority)
                
                for cw, tw in zip(corrected_words, true_words):
                    total_words += 1
                    if cw.lower() == tw.lower():
                        correct_words += 1
    char_acc = correct_chars / total_chars if total_chars > 0 else 0
    word_acc = correct_words / total_words if total_words > 0 else 0
    return char_acc, word_acc

# --- Training Loop ---
model_lstm = DiacNetCharModel(len(char_vocab)).to(device)
optimizer = optim.AdamW(model_lstm.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5, mode='max')
criterion = nn.CrossEntropyLoss(ignore_index=-100)

best_val_word_acc = 0.0
best_state        = None
no_improve        = 0

print(f"Training Character-level BiLSTM Model (up to {EPOCHS} epochs)...\n")
for epoch in range(1, EPOCHS + 1):
    model_lstm.train()
    total_loss = 0.0
    for padded_chars, lengths, labels, _ in train_loader:
        padded_chars = padded_chars.to(device)
        labels       = labels.to(device)
        optimizer.zero_grad()
        logits = model_lstm(padded_chars, lengths)
        loss = criterion(logits.view(-1, 6), labels.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    val_char_acc, val_word_acc = evaluate_char_model(model_lstm, val_loader, char_vocab, word_candidates)
    scheduler.step(val_word_acc)
    print(f"Epoch {epoch:2d} | loss: {total_loss/len(train_loader):.4f} | val_char_acc: {val_char_acc*100:.2f}% | val_word_acc: {val_word_acc*100:.2f}%")
    if val_word_acc > best_val_word_acc:
        best_val_word_acc = val_word_acc
        best_state        = {k: v.cpu().clone() for k, v in model_lstm.state_dict().items()}
        no_improve        = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

# --- Final Evaluation ---
model_lstm.load_state_dict(best_state)
test_char_acc, test_word_acc = evaluate_char_model(model_lstm, test_loader, char_vocab, word_candidates)
print(f"\nBest BiLSTM Val Word Accuracy: {best_val_word_acc*100:.2f}%")
print(f"BiLSTM Test Char Accuracy:     {test_char_acc*100:.2f}%")
print(f"BiLSTM Test Word Accuracy:     {test_word_acc*100:.2f}%")

# --- Save ---
torch.save({
    'model_state_dict': best_state,
    'char_vocab':       char_vocab,
    'config': {
        'emb_dim':    CHAR_EMB_DIM,
        'hidden_dim': HIDDEN_DIM,
        'model_type': 'char_bilstm'
    }
}, "diacnet_yor.pt")
with open("diacnet_yor_vocab.json", 'w', encoding='utf-8') as f:
    json.dump({'char_vocab': char_vocab, 'word_candidates': word_candidates}, f, ensure_ascii=False, indent=2)
print("Saved optimized Character-level BiLSTM model to diacnet_yor.pt and vocab to diacnet_yor_vocab.json")

## 4. Yoruba Transformer Diacritizer (`DiacNetYorX`)

This model fine-tunes `castorini/afriberta_large` as a word-level sequence labeler, which uses multilingual African contextualized embeddings for high accuracy.

In [ ]:
import json
import re
from collections import Counter, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def tokenize(text):
    return re.findall(r'\S+', text)

# Load corpus
with open("yoruba_diacritizer_corpus.json", 'r', encoding='utf-8') as f:
    corpus = json.load(f)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

BASE_MODEL = "castorini/afriberta_large"
BATCH_SIZE_X = 16
EPOCHS_X = 10
PATIENCE_X = 3
LR_ENCODER = 2e-5
LR_HEAD = 1e-3
WARMUP_RATIO = 0.1
MAX_SEQ_LEN = 128
PAD_LABEL_IDX = -100
MAX_CANDIDATES = 8
MIN_CAND_FREQ = 2

print(f"Loading tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def build_word_candidates(pairs):
    cand_counter = defaultdict(Counter)
    for pair in pairs:
        plain_words = tokenize(pair['plain'])
        diac_words  = tokenize(pair['diacritized'])
        if len(plain_words) != len(diac_words):
            continue
        for pw, dw in zip(plain_words, diac_words):
            cand_counter[pw.lower()][dw.lower()] += 1

    word_candidates = {}
    for pw_l, counts in cand_counter.items():
        cands = [dw for dw, cnt in counts.most_common(MAX_CANDIDATES) if cnt >= MIN_CAND_FREQ]
        if not cands:
            cands = [counts.most_common(1)[0][0]]
        if pw_l not in cands:
            cands.append(pw_l)
        word_candidates[pw_l] = cands
    return word_candidates

word_candidates_x = build_word_candidates(corpus['train'] + corpus['val'])
print(f"Transformer word candidates entries: {len(word_candidates_x)}")

class DiacNetXDataset(Dataset):
    def __init__(self, pairs, tokenizer, word_candidates):
        self.tokenizer = tokenizer
        self.word_candidates = word_candidates
        self.samples = []
        for pair in pairs:
            plain_words = tokenize(pair['plain'])
            diac_words  = tokenize(pair['diacritized'])
            if len(plain_words) != len(diac_words):
                continue
            self.samples.append((plain_words, diac_words))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn_x(batch, tokenizer, word_candidates, max_len=MAX_SEQ_LEN):
    plain_batch, diac_batch = zip(*batch)
    all_input_ids, all_attention, all_word_ids, all_labels = [], [], [], []

    for plain_words, diac_words in zip(plain_batch, diac_batch):
        encoding = tokenizer(
            plain_words,
            is_split_into_words=True,
            max_length=max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        word_ids = encoding.word_ids(batch_index=0)

        label_ids = []
        prev_word_idx = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(PAD_LABEL_IDX)
            elif wid != prev_word_idx:
                if wid < len(diac_words):
                    pw = plain_words[wid].lower()
                    dw = diac_words[wid].lower()
                    cands = word_candidates.get(pw, [pw])
                    try:
                        lbl = cands.index(dw)
                        if lbl >= MAX_CANDIDATES:
                            lbl = PAD_LABEL_IDX
                    except ValueError:
                        lbl = PAD_LABEL_IDX
                else:
                    lbl = PAD_LABEL_IDX
                label_ids.append(lbl)
                prev_word_idx = wid
            else:
                label_ids.append(PAD_LABEL_IDX)

        all_input_ids.append(input_ids)
        all_attention.append(attention_mask)
        all_word_ids.append(word_ids)
        all_labels.append(torch.tensor(label_ids, dtype=torch.long))

    return (
        torch.stack(all_input_ids),
        torch.stack(all_attention),
        all_word_ids,
        torch.stack(all_labels),
        list(plain_batch),
        list(diac_batch),
    )

def make_collate(tokenizer, word_candidates):
    return lambda batch: collate_fn_x(batch, tokenizer, word_candidates)

train_ds_x = DiacNetXDataset(corpus['train'], tokenizer, word_candidates_x)
val_ds_x   = DiacNetXDataset(corpus['val'],   tokenizer, word_candidates_x)
test_ds_x  = DiacNetXDataset(corpus['test'],  tokenizer, word_candidates_x)

collate_x = make_collate(tokenizer, word_candidates_x)
train_loader_x = DataLoader(train_ds_x, batch_size=BATCH_SIZE_X, shuffle=True,  collate_fn=collate_x)
val_loader_x   = DataLoader(val_ds_x,   batch_size=BATCH_SIZE_X, shuffle=False, collate_fn=collate_x)
test_loader_x  = DataLoader(test_ds_x,  batch_size=BATCH_SIZE_X, shuffle=False, collate_fn=collate_x)

class DiacNetYorXModel(nn.Module):
    def __init__(self, model_name=BASE_MODEL):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, MAX_CANDIDATES)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)
        logits  = self.classifier(seq_out)
        return logits

model_x = DiacNetYorXModel().to(device)

optimizer_x = optim.AdamW([
    {'params': model_x.encoder.parameters(),    'lr': LR_ENCODER},
    {'params': model_x.classifier.parameters(), 'lr': LR_HEAD},
])

total_steps = len(train_loader_x) * EPOCHS_X
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler_x = get_linear_schedule_with_warmup(optimizer_x, warmup_steps, total_steps)
criterion_x = nn.CrossEntropyLoss(ignore_index=PAD_LABEL_IDX)

def word_accuracy_from_batch(logits, labels, word_ids_batch, plain_batch, diac_batch, word_candidates):
    preds = logits.argmax(dim=-1)
    correct = total = 0
    for i, (wids, plain_words, diac_words) in enumerate(zip(word_ids_batch, plain_batch, diac_batch)):
        prev_wid = None
        for j, wid in enumerate(wids):
            if wid is None or wid == prev_wid:
                continue
            prev_wid = wid
            if wid >= len(diac_words):
                continue
            pw = plain_words[wid].lower()
            true_label = diac_words[wid].lower()
            cands = word_candidates.get(pw, [pw])
            pred_idx = preds[i, j].item()
            if pred_idx < len(cands):
                pred_label = cands[pred_idx]
            else:
                pred_label = cands[0]
            total += 1
            if pred_label == true_label:
                correct += 1
    return correct, total

def evaluate_x(model, loader, word_candidates):
    model.eval()
    total_correct = total_words = 0
    with torch.no_grad():
        for input_ids, attn_mask, word_ids_batch, labels, plain_batch, diac_batch in loader:
            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)
            labels    = labels.to(device)
            logits    = model(input_ids, attn_mask)
            c, t = word_accuracy_from_batch(
                logits.cpu(), labels.cpu(), word_ids_batch, plain_batch, diac_batch, word_candidates
            )
            total_correct += c
            total_words   += t
    return total_correct / total_words if total_words > 0 else 0

# Phase 1: Train Head Only (Encoder Frozen) - 3 epochs
print("Phase 1: Training Head Only (Encoder Frozen)...")
for param in model_x.encoder.parameters():
    param.requires_grad = False

for epoch in range(1, 4):
    model_x.train()
    total_loss = 0.0
    for input_ids, attn_mask, _, labels, _, _ in train_loader_x:
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels    = labels.to(device)

        optimizer_x.zero_grad()
        logits = model_x(input_ids, attn_mask)
        loss = criterion_x(logits.view(-1, logits.shape[-1]), labels.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_x.parameters(), 1.0)
        optimizer_x.step()
        scheduler_x.step()
        total_loss += loss.item()

    val_acc = evaluate_x(model_x, val_loader_x, word_candidates_x)
    print(f"Phase 1 Epoch {epoch} | loss: {total_loss/len(train_loader_x):.4f} | val_acc: {val_acc*100:.2f}%")

# Phase 2: Full Fine-tuning
print("\nPhase 2: Full Fine-tuning...")
for param in model_x.encoder.parameters():
    param.requires_grad = True

best_val_acc_x = 0.0
best_state_x   = None
no_improve_x   = 0

for epoch in range(1, EPOCHS_X + 1):
    model_x.train()
    total_loss = 0.0
    for input_ids, attn_mask, _, labels, _, _ in train_loader_x:
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels    = labels.to(device)

        optimizer_x.zero_grad()
        logits = model_x(input_ids, attn_mask)
        loss = criterion_x(logits.view(-1, logits.shape[-1]), labels.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_x.parameters(), 1.0)
        optimizer_x.step()
        scheduler_x.step()
        total_loss += loss.item()

    val_acc = evaluate_x(model_x, val_loader_x, word_candidates_x)
    print(f"Epoch {epoch:2d} | loss: {total_loss/len(train_loader_x):.4f} | val_acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc_x:
        best_val_acc_x = val_acc
        best_state_x   = {k: v.cpu().clone() for k, v in model_x.state_dict().items()}
        no_improve_x   = 0
    else:
        no_improve_x += 1
        if no_improve_x >= PATIENCE_X:
            print("Early stopping triggered.")
            break

model_x.load_state_dict(best_state_x)
test_acc_x = evaluate_x(model_x, test_loader_x, word_candidates_x)
print(f"\nBest Transformer Val Accuracy: {best_val_acc_x*100:.2f}%")
print(f"Transformer Test Accuracy:     {test_acc_x*100:.2f}%")

# Save
torch.save({
    'model_state_dict': best_state_x,
    'word_candidates':  word_candidates_x,
    'base_model':       BASE_MODEL,
    'num_labels':       MAX_CANDIDATES
}, "diacnet_yor_x.pt")
with open("diacnet_yor_x_vocab.json", 'w', encoding='utf-8') as f:
    json.dump({'word_candidates': word_candidates_x, 'base_model': BASE_MODEL}, f, ensure_ascii=False, indent=2)
print("Saved Transformer model to diacnet_yor_x.pt")

## 5. Igbo CRF Diacritizer (`DiacNetIbo`)

Restores Igbo dot-below vowels (`ị`, `ụ`, `ọ`, `ẹ`). Character-level features are processed with a fast CRF sequence labeler.

In [ ]:
import re
import json
import pickle
import unicodedata
from collections import Counter
import sklearn_crfsuite

def tokenize(text):
    return re.findall(r'\S+', text)

def strip_all_diacritics(text):
    decomposed = unicodedata.normalize('NFD', text)
    filtered = "".join(c for c in decomposed if unicodedata.category(c) != 'Mn')
    return unicodedata.normalize('NFC', filtered)

def word_features(words, i):
    word = words[i].lower()
    features = {
        'bias': 1.0,
        'word': word,
        'word[-2:]': word[-2:],
        'word[-3:]': word[-3:],
        'word[:2]':  word[:2],
        'word[:3]':  word[:3],
        'word.len':  str(min(len(word), 10)),
        'is_first':  i == 0,
        'is_last':   i == len(words) - 1,
        'is_upper':  words[i][0].isupper() if words[i] else False,
    }
    for j, ch in enumerate(word[:6]):
        features[f'ch[{j}]'] = ch

    if i > 0:
        prev = words[i-1].lower()
        features.update({'prev_word': prev, 'prev[-2:]': prev[-2:], 'prev[:2]': prev[:2]})
    else:
        features['BOS'] = True

    if i > 1:
        features['prev2_word'] = words[i-2].lower()

    if i < len(words) - 1:
        nxt = words[i+1].lower()
        features.update({'next_word': nxt, 'next[-2:]': nxt[-2:], 'next[:2]': nxt[:2]})
    else:
        features['EOS'] = True

    if i < len(words) - 2:
        features['next2_word'] = words[i+2].lower()

    return features

def build_dataset(pairs):
    X, y = [], []
    for pair in pairs:
        plain_words = tokenize(pair['plain'])
        diac_words  = tokenize(pair['diacritized'])
        if len(plain_words) != len(diac_words):
            continue
        X.append([word_features(plain_words, i) for i in range(len(plain_words))])
        y.append(list(diac_words))
    return X, y

def word_accuracy(crf, X, y_true):
    y_pred = crf.predict(X)
    correct = total = 0
    for true_seq, pred_seq in zip(y_true, y_pred):
        for t, p in zip(true_seq, pred_seq):
            total += 1
            if t.lower() == p.lower():
                correct += 1
    return correct / total if total > 0 else 0

with open("igbo_diacritizer_corpus.json", 'r', encoding='utf-8') as f:
    ibo_corpus = json.load(f)

X_train_ib, y_train_ib = build_dataset(ibo_corpus['train'])
X_val_ib,   y_val_ib   = build_dataset(ibo_corpus['val'])
X_test_ib,  y_test_ib  = build_dataset(ibo_corpus['test'])

vocab_ib = {}
for pair in ibo_corpus['train'] + ibo_corpus['val']:
    plain_words = tokenize(pair['plain'])
    diac_words  = tokenize(pair['diacritized'])
    if len(plain_words) != len(diac_words):
        continue
    for pw, dw in zip(plain_words, diac_words):
        key = pw.lower()
        if key not in vocab_ib:
            vocab_ib[key] = Counter()
        vocab_ib[key][dw.lower()] += 1
vocab_top_ib = {k: [c for c, _ in v.most_common(3)] for k, v in vocab_ib.items()}

print("Training Igbo CRF (on 1,000-sentence subset to prevent vocab scaling slow-down)...\n")
crf_ibo = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=50,
    all_possible_transitions=False,
    verbose=True,
)
crf_ibo.fit(X_train_ib[:1000], y_train_ib[:1000])

val_word_ib = word_accuracy(crf_ibo, X_val_ib, y_val_ib)
test_word_ib = word_accuracy(crf_ibo, X_test_ib, y_test_ib)
print(f"\nIgbo CRF Validation Word Accuracy: {val_word_ib*100:.2f}%")
print(f"Igbo CRF Test Word Accuracy:       {test_word_ib*100:.2f}%")

# Save
with open("diacnet_ibo.pkl", 'wb') as f:
    pickle.dump({'crf': crf_ibo, 'vocab': vocab_top_ib}, f)
with open("diacnet_ibo_vocab.json", 'w', encoding='utf-8') as f:
    json.dump(vocab_top_ib, f, ensure_ascii=False, indent=2)
print("Saved Igbo model to diacnet_ibo.pkl and vocab to diacnet_ibo_vocab.json")

## 6. Inference Demo & Google Drive Export

Test predictions on sample strings and download/export all trained model files to Google Drive.

In [ ]:
def run_crf_inference(crf, vocab_top, text):
    words = tokenize(text)
    feats = [word_features(words, i) for i in range(len(words))]
    pred = crf.predict([feats])[0]
    return " ".join(pred)

print("--- Yoruba Dot-Below CRF Demo ---")
print(f"Input:  ojo lo si oja lana")
print(f"Output: {run_crf_inference(crf_yor_db, vocab_top, 'ojo lo si oja lana')}\n")

print("--- Igbo CRF Demo ---")
print(f"Input:  kedu ka i mere")
print(f"Output: {run_crf_inference(crf_ibo, vocab_top_ib, 'kedu ka i mere')}\n")

# Save files to Google Drive
from google.colab import drive
try:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/olaverse_models
    !cp diacnet_yor_db.pkl diacnet_yor_db_vocab.json diacnet_yor.pt diacnet_yor_vocab.json diacnet_yor_x.pt diacnet_yor_x_vocab.json diacnet_ibo.pkl diacnet_ibo_vocab.json /content/drive/MyDrive/olaverse_models/
    print("\n✅ All model outputs have been successfully backed up to Google Drive under 'olaverse_models'!")
except Exception as e:
    print(f"Could not automatically save to Google Drive (skip if not running in Google Colab): {e}")